# 🧠 Engram — Persistent KV-Cache Inference Server

> Named, tiered, persistent KV-cache sessions for LLM inference.  
> Only the *delta* tokens are prefilled on each turn — not the full history.

## What this notebook does
1. Checks GPU availability
2. Installs build dependencies
3. Clones and builds Engram with CUDA support
4. Downloads a GGUF model from Hugging Face
5. Starts the Engram server in the background
6. Runs a **baseline** benchmark (Ollama-style: full history every turn)
7. Runs an **Engram chat** benchmark (delta-only prefill)
8. Prints a side-by-side comparison

**Runtime:** GPU (T4/A100). Change to CPU if needed but expect ~10× slower decode.

---
### ⚠️ Before running
- Go to **Runtime → Change runtime type → T4 GPU**
- Run cells top-to-bottom (Ctrl+F9 to run all)
- Build takes ~5 min on first run (compiling llama.cpp with CUDA)

## 0. Check GPU

In [ ]:
!nvidia-smi
import subprocess, sys
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    print(f"\n✅ GPU found: {result.stdout.strip()}")
else:
    print("⚠️  No GPU detected — set Runtime → Change runtime type → GPU")

## 1. Install build dependencies

In [ ]:
%%bash
set -e
apt-get update -qq
apt-get install -y -qq cmake ninja-build git build-essential curl wget
echo "✅ Build dependencies installed"
cmake --version | head -1
nvcc --version | head -4

## 2. Clone and build Engram (CUDA)

In [ ]:
%%bash
set -e
cd /content

if [ -d engram ]; then
  echo "Engram already cloned — pulling latest"
  cd engram && git pull --ff-only
else
  git clone https://github.com/pilsy/engram.git
  cd engram
fi

echo "✅ Engram cloned"
git log --oneline -5

In [ ]:
%%bash
set -e
cd /content/engram

# Configure with CUDA enabled
cmake -B build \
  -DCMAKE_BUILD_TYPE=Release \
  -DGGML_CUDA=ON \
  -GNinja

echo "\n🔨 Building (this takes ~5 min on first run)..."
cmake --build build -j$(nproc) 2>&1 | tail -20

echo "\n✅ Build complete"
ls -lh build/engram

## 3. Download model

Using **Mistral 7B Instruct v0.2 Q4_K_M** (~4.1 GB) from Hugging Face.  
Change `MODEL_URL` and `MODEL_FILE` to use a different model.

In [ ]:
import os

# ── Model config — change these to use a different model ──────────────────
MODEL_REPO = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
MODEL_FILE = "mistral-7b-instruct-v0.2.Q4_K_M.gguf"
MODEL_PATH = f"/content/models/{MODEL_FILE}"
# ──────────────────────────────────────────────────────────────────────────

os.makedirs("/content/models", exist_ok=True)
print(f"Model: {MODEL_REPO}/{MODEL_FILE}")
print(f"Destination: {MODEL_PATH}")

In [ ]:
%%bash -s "{MODEL_REPO}" "{MODEL_FILE}" "{MODEL_PATH}"
REPO=$1
FILE=$2
DEST=$3

if [ -f "$DEST" ]; then
  echo "✅ Model already downloaded: $(du -h $DEST | cut -f1)"
  exit 0
fi

URL="https://huggingface.co/$REPO/resolve/main/$FILE"
echo "Downloading from $URL"
wget -q --show-progress -O "$DEST" "$URL"
echo "✅ Downloaded: $(du -h $DEST | cut -f1)"

## 4. Start the Engram server

In [ ]:
import subprocess, time, requests, os, signal

MODEL_PATH = "/content/models/mistral-7b-instruct-v0.2.Q4_K_M.gguf"
ENGRAM_HOST = "http://127.0.0.1:8080"

# Kill any existing engram process
subprocess.run(["pkill", "-f", "engram"], capture_output=True)
time.sleep(1)

os.makedirs("/content/sessions", exist_ok=True)

# Detect GPU layer count (use all layers for GPU, 0 for CPU)
try:
    gpu_info = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits'],
        text=True
    ).strip()
    vram_mb = int(gpu_info.split('\n')[0])
    # Rough heuristic: Mistral 7B Q4_K_M needs ~4.1 GB for weights
    # leave 1 GB headroom for KV cache
    GPU_LAYERS = 32 if vram_mb >= 6000 else 16 if vram_mb >= 4000 else 0
    print(f"GPU VRAM: {vram_mb} MB → using {GPU_LAYERS} GPU layers")
except Exception:
    GPU_LAYERS = 0
    print("No GPU detected → CPU-only mode")

# Start server
log = open("/content/engram.log", "w")
proc = subprocess.Popen(
    [
        "/content/engram/build/engram",
        "--host", "127.0.0.1",
        "--port", "8080",
        "--model", MODEL_PATH,
        "--max-hot", "2",
        "--max-warm", "8",
        "--sessions-dir", "/content/sessions",
        "--threads", "4",
        "--gpu-layers", str(GPU_LAYERS),
    ],
    stdout=log, stderr=log
)

print(f"Engram PID: {proc.pid} — waiting for server to be ready...")

for i in range(60):
    time.sleep(2)
    try:
        r = requests.get(f"{ENGRAM_HOST}/sessions", timeout=2)
        if r.status_code == 200:
            print(f"\n✅ Engram is up! ({r.json()})")
            break
    except Exception:
        print(".", end="", flush=True)
else:
    print("\n❌ Server did not start — check logs below")
    with open("/content/engram.log") as f:
        print(f.read()[-3000:])

In [ ]:
# Optional: tail the server log
!tail -40 /content/engram.log

## 5. Benchmark helpers

In [ ]:
import requests, time, json
from datetime import datetime

ENGRAM_HOST = "http://127.0.0.1:8080"
W = 74

PROMPTS = [
    "What is a KV cache in the context of transformer models?",
    "How does prefill differ from decode in LLM inference?",
    "Why does prefill cost scale quadratically with context length?",
    "What would happen if you could skip prefill on subsequent calls?",
    "How might you serialize a KV cache to disk and restore it later?",
    "What tradeoffs exist between VRAM, RAM, and NVMe for cache storage?",
    "How would you handle context window overflow in a persistent cache server?",
    "What would a benchmark harness for this look like?",
    "How does Flash Attention interact with a persistent KV cache design?",
    "What are the most impactful use cases for persistent KV caches in agentic workflows?",
]

def fmt_ms(ms):
    if ms is None: return "n/a"
    return f"{ms/1000:.2f}s" if ms >= 1000 else f"{ms:.0f}ms"

def bar(val, max_val, width=28, fill="█", empty="░"):
    n = round((val / max_val) * width) if max_val > 0 else 0
    return fill * n + empty * (width - n)

def engram_post(path, body=None):
    r = requests.post(f"{ENGRAM_HOST}{path}",
                      json=body,
                      headers={"Content-Type": "application/json"},
                      timeout=300)
    r.raise_for_status()
    return r.json()

def engram_get(path):
    r = requests.get(f"{ENGRAM_HOST}{path}", timeout=30)
    r.raise_for_status()
    return r.json()

def engram_delete(path, body=None):
    r = requests.delete(f"{ENGRAM_HOST}{path}",
                        json=body,
                        headers={"Content-Type": "application/json"},
                        timeout=30)
    r.raise_for_status()
    return r.json()

print("✅ Helpers loaded")

## 6. Baseline benchmark

Simulates what a standard inference server does: re-sends the **full conversation history** on every call.  
Prefill grows O(n²) — each turn processes all prior turns from scratch.

In [ ]:
N_TURNS    = 10
N_PREDICT  = 128
N_CTX      = 4096
GPU_LAYERS = 32  # set to 0 if CPU-only

print("╔══════════════════════════════════════════════╗")
print("║    standard baseline — full history / turn   ║")
print("╚══════════════════════════════════════════════╝")
print(f"  turns: {N_TURNS}  n_predict: {N_PREDICT}  gpu_layers: {GPU_LAYERS}")
print()
print("  Simulates O(n²) prefill: each turn resends the entire history.")
print()

SESSION_ID = f"baseline-{int(time.time())}"

# Create session
engram_post("/session/create", {
    "session_id": SESSION_ID,
    "n_ctx": N_CTX,
    "n_gpu_layers": GPU_LAYERS,
})

print("-" * W)
print(f" {'Turn':<5} {'Prompt tok':<14} {'Prefill':<12} {'Decode':<12} {'Total':<10} Slowdown")
print("-" * W)

messages = []
baseline_results = []
first_prefill = None

for i in range(N_TURNS):
    user_msg = PROMPTS[i % len(PROMPTS)]
    messages.append({"role": "user", "content": user_msg})

    # Send full history — this is the O(n²) baseline
    t0 = time.time()
    r = engram_post("/session/chat", {
        "session_id": SESSION_ID,
        "messages": list(messages),
        "n_predict": N_PREDICT,
        "temperature": 0.7,
        "top_p": 0.9,
        "top_k": 40,
    })
    wall_ms = (time.time() - t0) * 1000

    if r.get("text"):
        messages.append({"role": "assistant", "content": r["text"]})

    ms_prefill = r.get("ms_prefill", 0)
    ms_decode  = r.get("ms_decode", 0)
    n_prompt   = r.get("n_tokens_prompt", 0)

    if first_prefill is None:
        first_prefill = ms_prefill

    slowdown = f"{ms_prefill / first_prefill:.1f}×" if first_prefill else "—"

    baseline_results.append({
        "turn": i + 1,
        "n_tokens_prompt": n_prompt,
        "ms_prefill": ms_prefill,
        "ms_decode": ms_decode,
        "wall_ms": wall_ms,
        "user": user_msg,
        "assistant": r.get("text", ""),
    })

    print(f" {i+1:<5} {n_prompt:<14} {fmt_ms(ms_prefill):<12} {fmt_ms(ms_decode):<12} {fmt_ms(wall_ms):<10} {slowdown}")

print("-" * W)

# Cleanup — evict so the session doesn't pollute the cache
engram_delete("/session/evict", {"session_id": SESSION_ID})

print()
first = baseline_results[0]
last  = baseline_results[-1]
print("Performance Summary (Baseline)")
print("─" * 34)
print(f"  Turn 1  prefill : {fmt_ms(first['ms_prefill'])}  ({first['n_tokens_prompt']} tokens)")
print(f"  Turn {N_TURNS} prefill : {fmt_ms(last['ms_prefill'])}  ({last['n_tokens_prompt']} tokens)")
if first['ms_prefill'] and last['ms_prefill']:
    print(f"  Slowdown (1→{N_TURNS}) : {last['ms_prefill']/first['ms_prefill']:.1f}×  ← this is what Engram eliminates")
print()
max_pre = max(r['ms_prefill'] for r in baseline_results)
for r in baseline_results:
    b = bar(r['ms_prefill'], max_pre)
    print(f"  [{r['turn']:2d}] {b} {fmt_ms(r['ms_prefill'])}  ({r['n_tokens_prompt']} tok)")

## 7. Engram chat benchmark

Uses `/session/chat` with the **same prompts and full message history** sent each turn.  
Engram's delta-aware inference extracts only the new tokens and prefills those — the rest is served from the persistent KV cache.

In [ ]:
print("╔══════════════════════════════════════════════╗")
print("║       engram chat — delta-only prefill       ║")
print("╚══════════════════════════════════════════════╝")
print(f"  turns: {N_TURNS}  n_predict: {N_PREDICT}  gpu_layers: {GPU_LAYERS}")
print()
print("  Same prompts. Same message history passed each turn.")
print("  Only delta tokens (new content) hit prefill.")
print()

SESSION_ID = f"engram-chat-{int(time.time())}"

engram_post("/session/create", {
    "session_id": SESSION_ID,
    "n_ctx": N_CTX,
    "n_gpu_layers": GPU_LAYERS,
})

print("-" * W)
print(f" {'Turn':<5} {'Cached tok':<12} {'Delta tok':<11} {'Prefill':<12} {'Decode':<12} {'Hit':<5} Speedup")
print("-" * W)

messages = []
engram_results = []
first_prefill = None

for i in range(N_TURNS):
    user_msg = PROMPTS[i % len(PROMPTS)]
    messages.append({"role": "user", "content": user_msg})

    r = engram_post("/session/chat", {
        "session_id": SESSION_ID,
        "messages": list(messages),
        "n_predict": N_PREDICT,
        "temperature": 0.7,
        "top_p": 0.9,
        "top_k": 40,
    })

    if r.get("text"):
        messages.append({"role": "assistant", "content": r["text"]})

    ms_prefill      = r.get("ms_prefill", 0)
    ms_decode       = r.get("ms_decode", 0)
    n_delta         = r.get("n_tokens_prompt", 0)
    n_cached        = r.get("n_tokens_in_cache", 0)
    cache_hit       = r.get("cache_hit", False)

    if first_prefill is None:
        first_prefill = ms_prefill

    speedup = f"{first_prefill / ms_prefill:.1f}×" if ms_prefill else "—"
    hit_mark = "✓" if cache_hit else "✗"

    engram_results.append({
        "turn": i + 1,
        "n_cached": n_cached,
        "n_delta": n_delta,
        "ms_prefill": ms_prefill,
        "ms_decode": ms_decode,
        "cache_hit": cache_hit,
        "user": user_msg,
        "assistant": r.get("text", ""),
    })

    print(f" {i+1:<5} {n_cached:<12} {n_delta:<11} {fmt_ms(ms_prefill):<12} {fmt_ms(ms_decode):<12} {hit_mark:<5} {speedup}")

print("-" * W)

# Session status before evict
status = engram_get(f"/session/status?session_id={SESSION_ID}")
engram_delete("/session/evict", {"session_id": SESSION_ID})

print()
first = engram_results[0]
last  = engram_results[-1]
print("Performance Summary (Engram Chat)")
print("─" * 35)
print(f"  Turn 1  prefill : {fmt_ms(first['ms_prefill'])}  ({first['n_delta']} delta tokens, cold start)")
print(f"  Turn {N_TURNS} prefill : {fmt_ms(last['ms_prefill'])}  ({last['n_delta']} delta tokens, cache={last['n_cached']})")
if first['ms_prefill'] and last['ms_prefill']:
    creep = last['ms_prefill'] / first['ms_prefill']
    print(f"  Creep (1→{N_TURNS})    : {creep:.1f}×  (expected: O(delta × cached) attention)")
print(f"  Total tokens in cache: {status.get('n_tokens_used', '?')}")
print()
max_pre = max(r['ms_prefill'] for r in engram_results)
for r in engram_results:
    b = bar(r['ms_prefill'], max_pre)
    print(f"  [{r['turn']:2d}] {b} {fmt_ms(r['ms_prefill'])}  (delta={r['n_delta']} cached={r['n_cached']})") 

## 8. Side-by-side comparison

In [ ]:
print("=" * W)
print("  SIDE-BY-SIDE: Baseline vs Engram Chat")
print("=" * W)
print(f"  {'Turn':<5} {'Baseline prefill':<20} {'Engram prefill':<20} {'Speedup'}")
print("-" * W)

speedups = []
for b, e in zip(baseline_results, engram_results):
    if e['ms_prefill'] > 0:
        speedup = b['ms_prefill'] / e['ms_prefill']
        speedups.append(speedup)
        speedup_str = f"{speedup:.1f}×"
    else:
        speedup_str = "—"
    print(f"  {b['turn']:<5} {fmt_ms(b['ms_prefill']):<20} {fmt_ms(e['ms_prefill']):<20} {speedup_str}")

print("=" * W)
print()

# Headline numbers
b1, bN = baseline_results[0], baseline_results[-1]
e1, eN = engram_results[0],   engram_results[-1]

baseline_slowdown = bN['ms_prefill'] / b1['ms_prefill'] if b1['ms_prefill'] else 0
engram_creep      = eN['ms_prefill'] / e1['ms_prefill'] if e1['ms_prefill'] else 0
final_speedup     = bN['ms_prefill'] / eN['ms_prefill'] if eN['ms_prefill'] else 0

print(f"  Baseline   turn 1 → {N_TURNS}: {fmt_ms(b1['ms_prefill'])} → {fmt_ms(bN['ms_prefill'])}  ({baseline_slowdown:.1f}× slowdown)")
print(f"  Engram     turn 1 → {N_TURNS}: {fmt_ms(e1['ms_prefill'])} → {fmt_ms(eN['ms_prefill'])}  ({engram_creep:.1f}× creep)")
print()
print(f"  ⚡ Engram is {final_speedup:.0f}× faster on prefill at turn {N_TURNS}")
print()
print(f"  The {baseline_slowdown:.0f}× baseline slowdown is eliminated.")
print(f"  The {engram_creep:.1f}× Engram creep is unavoidable physics:")
print(f"    each new token attends to all {eN['n_cached']} cached keys/values.")
print(f"    formula: prefill ∝ delta_tokens × cached_tokens")
print()

# Simple ASCII chart
print("  Prefill latency per turn:")
print()
all_prefills = [r['ms_prefill'] for r in baseline_results + engram_results]
max_pre = max(all_prefills)
print(f"  {'':4} {'Baseline':^32} {'Engram':^32}")
for b, e in zip(baseline_results, engram_results):
    bb = bar(b['ms_prefill'], max_pre, 28)
    eb = bar(e['ms_prefill'], max_pre, 28)
    print(f"  [{b['turn']:2d}] {bb} {eb}")
print(f"       0{'':26}{fmt_ms(max_pre)}")

## 9. Conversation transcript (Engram)

In [ ]:
def word_wrap(text, width=64):
    lines = []
    for para in text.split("\n"):
        if len(para) <= width:
            lines.append(para)
            continue
        words = para.split()
        cur = ""
        for w in words:
            if not cur:
                cur = w
            elif len(cur) + 1 + len(w) <= width:
                cur += " " + w
            else:
                lines.append(cur)
                cur = w
        if cur:
            lines.append(cur)
    return lines or [""]

print("━" * W)
print("  CONVERSATION TRANSCRIPT (Engram chat)")
print("━" * W)

for r in engram_results:
    hit = "✓ cache hit" if r['cache_hit'] else "✗ cold start"
    print()
    print(f"  ┌─ Turn {r['turn']}  prefill={fmt_ms(r['ms_prefill'])}  decode={fmt_ms(r['ms_decode'])}  {hit}")
    print(f"  │  delta={r['n_delta']} tok  cached={r['n_cached']} tok")
    print(f"  │")
    print(f"  │  👤 User:")
    for line in word_wrap(r['user']):
        print(f"  │     {line}")
    print(f"  │")
    print(f"  │  🤖 Assistant:")
    reply = r['assistant'].strip() or "(no response)"
    for line in word_wrap(reply):
        print(f"  │     {line}")
    print(f"  └" + "─" * (W - 4))

print()
print("━" * W)

## 10. Tear down

Stop the Engram server and clean up session blobs.

In [ ]:
import subprocess
subprocess.run(["pkill", "-f", "engram"], capture_output=True)
print("✅ Engram server stopped")
!rm -rf /content/sessions
print("✅ Session blobs cleaned up")